# Import FRED data
- Author: Bryan Bravo
- Created: 2026-03-02
## Import Libraries

In [2]:
import os
import sys

# Set JAVA_HOME before importing PySpark and use findspark
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-22'  # May need to remove or update in cloud environment.
import findspark
findspark.init()

import requests
import pandas as pd
import numpy as np
import json
import pyspark
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta
from functools import reduce
from pyspark.sql import (
    functions as F,
    Window as W,
    types as T,
    SparkSession,
    DataFrame
)

# api keys and other hardcoded values for the Strait of Hormuz Research project.
# Note: In a production environment, these should be stored securely and not hardcoded.
import hardcoded_keys # Note: This file is added to .gitignore to prevent accidental commits of sensitive information.

### Initialize Spark Session


In [3]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("BusinessPlanAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.parquet.nativeio.enabled", "false") \
    .getOrCreate()

print("Spark Session created successfully!")

Spark Session created successfully!


### Custom Functions

In [ ]:
def fetch_fred_data(
    api_key: str,
    currency_name: str, # Currency name of series id.
    series_id: str,  # FRED series ID
    rate_name: str = None,
    start_date: str = "2006-04-01",  # Chosen because the broad fx rate for the US begins in 2006
    end_date: str = (dt.now().replace(day=1) - relativedelta(days=1)).strftime("%Y-%m-%d")
    ) -> pd.DataFrame:
    """
    Fetches daily foreign exchange rate data from the FRED API, cleans it, and returns
    a standardized DataFrame containing the latest revision for each date.

    This function:
    - Adds a `currency` column for identification
    - Computes a unified USD exchange rate (`us_fx_rate`) so that:
        • If the series is FX-per-USD (e.g., DEXJPUS), it inverts the value
        • Otherwise, it uses the value as-is

    Parameters
    ----------
    api_key : str
        FRED API key used for authentication.
    currency_name : str
        Human-readable currency name to attach to the output (e.g., "euro", "yen").
    series_id : str
        FRED series ID (e.g., "DEXUSEU", "DEXJPUS").
    rate_name : str
        If API value is based on interest rates, this will create a column name matching the string value.
    start_date : str
        Start date for the query in YYYY-MM-DD format. (Defaults to "2006-01-01")
    end_date : str
        End date for the query in YYYY-MM-DD format. (Defaults to the last day of the previous month)

    Returns
    -------
    pd.DataFrame
        A DataFrame with columns:
        - date : datetime64
        - currency : str
        - `rate_name` <- value derived from variable given : float
        representing the latest available FX rate for each date.

    Raises
    ------
    Exception
        If the API request fails or the response cannot be parsed.

    """

    try:
        response = requests.get(
            f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&realtime_start={start_date}&realtime_end={end_date}&api_key={api_key}&file_type=json",
            timeout=10)
        fred_data = response.json()

        if 'error_code' in fred_data:  # check if response is None
            print(f"✗ FRED API error: {fred_data.get('error_code', 'error_message')}")

        # Create pandas DataFrame from observations
        df = pd.DataFrame(fred_data['observations'])

        # Select relevant columns
        df = df[['date', 'realtime_start', 'value']]
        df['currency'] = currency_name


        # Correct data types and add joining variables
        df['date'] = pd.to_datetime(df['date'])
        df['realtime_start'] = pd.to_datetime(df['realtime_start'])
        df['value'] = pd.to_numeric(df['value'], errors='coerce')  # Convert to numeric, coerce errors to NaN

        df['join_dt'] = df['date'].dt.strftime("%Y%m").astype(int)
        # Keep only the most recent revision for each date
        df = df.loc[df.groupby('date')['realtime_start'].idxmax()]
        df = df[df['date'] >= start_date]

        # Add USD exchange rate (USD/FX)
        if rate_name == 'fx_rate':
            if series_id[5:] == "US":
                df['fx_rate'] = 1 / df['value']
            else:
                df['fx_rate'] = df['value']

            # subset columns
            df = df[['date', 'currency', 'fx_rate', 'join_dt']].reset_index(drop=True)

        # Add column with identified rate name
        else:
            df[rate_name] = df['value']
            # subset columns
            df = df[['date', 'currency', rate_name, 'join_dt']].reset_index(drop=True)
        

        print(f"""
    ✓ Fetched {len(df)} records for '{currency_name}': {rate_name}
    from FRED for years {dt.strftime(df['date'].min(), '%Y-%m-%d')} through {dt.strftime(df['date'].max(), '%Y-%m-%d')}
    ******************************************************************************""")
        return df
    except Exception as e:
        print(f"✗ Error fetching FRED data on {currency_name}: {str(e)[:100]}")

### Variables

In [5]:
fred_api = hardcoded_keys.FRED_API_KEY

## Query

In [ ]:
fred_fx_series = {
    "euro": {'interest_rate': 'IR3TIB01EZM156N', 'fx_rate': 'DEXUSEU'},
    # Organization for Economic Co-operation and Development, Interest Rates: 3-Month or 90-Day Rates and Yields: Interbank Rates: Total for Euro Area (19 Countries) [IR3TIB01EZM156N], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/IR3TIB01EZM156N, March 14, 2026.
    # Board of Governors of the Federal Reserve System (US), U.S. Dollars to Euro Spot Exchange Rate [DEXUSEU], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/DEXUSEU, March 14, 2026.
    "japan": {'interest_rate': 'INTGSTJPM193N', 'fx_rate': 'DEXJPUS'},
    # International Monetary Fund, Interest Rates, Government Securities, Treasury Bills for Japan [INTGSTJPM193N], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/INTGSTJPM193N, March 14, 2026.
    # Board of Governors of the Federal Reserve System (US), Japanese Yen to U.S. Dollar Spot Exchange Rate [DEXJPUS], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/DEXJPUS, March 14, 2026.
    "china": {'interest_rate': 'IR3TIB01CNM156N', 'fx_rate': "DEXCHUS"},
    # Organization for Economic Co-operation and Development, Interest Rates: 3-Month or 90-Day Rates and Yields: Interbank Rates: Total for China [IR3TIB01CNM156N], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/IR3TIB01CNM156N, March 14, 2026.
    # Board of Governors of the Federal Reserve System (US), Chinese Yuan Renminbi to U.S. Dollar Spot Exchange Rate [DEXCHUS], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/DEXCHUS, March 14, 2026.
    "canada": {'interest_rate': 'INTGSTCAM193N', 'fx_rate': 'DEXCAUS'},
    # International Monetary Fund, Interest Rates, Government Securities, Treasury Bills for Canada [INTGSTCAM193N], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/INTGSTCAM193N, March 14, 2026.
    # Board of Governors of the Federal Reserve System (US), Canadian Dollars to U.S. Dollar Spot Exchange Rate [DEXCAUS], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/DEXCAUS, March 14, 2026.
    "united_kingdom": {'interest_rate': 'INTGSTGBM193N', 'fx_rate': 'DEXUSUK'},
    # International Monetary Fund, Interest Rates, Government Securities, Treasury Bills for United Kingdom [INTGSTGBM193N], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/INTGSTGBM193N, March 14, 2026.
    # Board of Governors of the Federal Reserve System (US), U.S. Dollars to U.K. Pound Sterling Spot Exchange Rate [DEXUSUK], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/DEXUSUK, March 14, 2026.
    "south_korea": {'interest_rate': 'IR3TIB01KRM156N', 'fx_rate': 'DEXKOUS'},
    # Organization for Economic Co-operation and Development, Interest Rates: 3-Month or 90-Day Rates and Yields: Interbank Rates: Total for Korea [IR3TIB01KRM156N], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/IR3TIB01KRM156N, March 14, 2026.
    # Board of Governors of the Federal Reserve System (US), South Korean Won to U.S. Dollar Spot Exchange Rate [DEXKOUS], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/DEXKOUS, March 14, 2026.
    "mexico": {'interest_rate': 'INTGSTMXM193N', 'fx_rate': 'DEXMXUS'},
    # International Monetary Fund, Interest Rates, Government Securities, Treasury Bills for Mexico [INTGSTMXM193N], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/INTGSTMXM193N, March 14, 2026.
    # Board of Governors of the Federal Reserve System (US), Mexican Pesos to U.S. Dollar Spot Exchange Rate [DEXMXUS], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/DEXMXUS, March 14, 2026.
    "india": {'interest_rate': 'INDIR3TIB01STM', 'fx_rate': 'DEXINUS'},
    # Organization for Economic Co-operation and Development, Interest Rates: 3-Month or 90-Day Rates and Yields: Interbank Rates: Total for India [INDIR3TIB01STM], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/INDIR3TIB01STM, March 14, 2026.
    # Board of Governors of the Federal Reserve System (US), Indian Rupees to U.S. Dollar Spot Exchange Rate [DEXINUS], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/DEXINUS, March 14, 2026.
    "brazil": {'interest_rate': 'INTGSTBRM193N', 'fx_rate': 'DEXBZUS'},
    # International Monetary Fund, Interest Rates, Government Securities, Treasury Bills for Brazil [INTGSTBRM193N], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/INTGSTBRM193N, March 14, 2026.
    # Board of Governors of the Federal Reserve System (US), Brazilian Reals to U.S. Dollar Spot Exchange Rate [DEXBZUS], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/DEXBZUS, March 14, 2026.
    "australia": {'interest_rate': 'INTGSTAUM193N', 'fx_rate': 'DEXUSAL'},
    # International Monetary Fund, Interest Rates, Government Securities, Treasury Bills for Australia [INTGSTAUM193N], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/INTGSTAUM193N, March 14, 2026.
    # Board of Governors of the Federal Reserve System (US), U.S. Dollars to Australian Dollar Spot Exchange Rate [DEXUSAL], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/DEXUSAL, March 14, 2026.
    "switzerland": {'interest_rate': 'IR3TIB01CHM156N', 'fx_rate': 'DEXSZUS'},
    # Organization for Economic Co-operation and Development, Interest Rates: 3-Month or 90-Day Rates and Yields: Interbank Rates: Total for Switzerland [IR3TIB01CHM156N], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/IR3TIB01CHM156N, March 14, 2026.
    # Board of Governors of the Federal Reserve System (US), Swiss Francs to U.S. Dollar Spot Exchange Rate [DEXSZUS], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/DEXSZUS, March 14, 2026.
    "sweden": {'interest_rate': 'IR3TIB01SEM156N', 'fx_rate': 'DEXSDUS'},
    "norway": {'interest_rate': 'IR3TIB01NOM156N', 'fx_rate': 'DEXNOUS'},
    "denmark": {'interest_rate': 'IR3TIB01DKM156N', 'fx_rate': 'DEXDNUS'},
    "south_africa": {'interest_rate': 'INTGSTZAM193N', 'fx_rate': "DEXSFUS"},
    "new_zealand": {'interest_rate': 'IR3TIB01NZM156N', 'fx_rate': "DEXUSNZ"},
    "united_states": {'interest_rate': 'INTGSTUSM193N', 'fx_rate': "DTWEXBGS"},
    "russia": {'interest_rate': 'INTGSTRUM193N', 'fx_rate': 'CCUSMA02RUM618N'} 
}

fred_df = pd.DataFrame([[]])
for country, kval in fred_fx_series.items():
    

    for i, (rate_name, series) in enumerate(kval.items()):
        if i == 0:
            df1 = fetch_fred_data(fred_api, country, series, rate_name)
        else:
            df2 = fetch_fred_data(fred_api, country, series, rate_name)
            df1 = (df1.drop('date', axis=1)
                   .merge(df2, how='outer', on=['join_dt', 'currency']))

        fred_df = pd.concat([fred_df, df1], ignore_index=True)

# Convert to PySpark df
fred_df = (
    spark.createDataFrame(fred_df.drop('join_dt', axis=1))
    .withColumn('date', F.date_format(F.col('date'), "yyyyMMdd").cast('int'))
    .dropna())
fred_df.cache().count()


    ✓ Fetched 286 records for 'euro': interest_rate
    from FRED for years 2002-04-01 through 2026-01-01
    ******************************************************************************

    ✓ Fetched 6235 records for 'euro': fx_rate
    from FRED for years 2002-04-01 through 2026-02-20
    ******************************************************************************

    ✓ Fetched 183 records for 'japan': interest_rate
    from FRED for years 2002-04-01 through 2017-06-01
    ******************************************************************************

    ✓ Fetched 6235 records for 'japan': fx_rate
    from FRED for years 2002-04-01 through 2026-02-20
    ******************************************************************************

    ✓ Fetched 285 records for 'china': interest_rate
    from FRED for years 2002-04-01 through 2025-12-01
    ******************************************************************************

    ✓ Fetched 6235 records for 'china': fx_rate
    from

85462

In [10]:
fred_df.show()

+--------+--------+----------------+-------+
|    date|currency|   interest_rate|fx_rate|
+--------+--------+----------------+-------+
|20020331|    euro|3.40690476190476| 0.8806|
|20020401|    euro|3.40690476190476| 0.8782|
|20020402|    euro|3.40690476190476| 0.8804|
|20020403|    euro|3.40690476190476| 0.8779|
|20020404|    euro|3.40690476190476| 0.8805|
|20020408|    euro|3.40690476190476|  0.875|
|20020409|    euro|3.40690476190476| 0.8793|
|20020410|    euro|3.40690476190476| 0.8794|
|20020411|    euro|3.40690476190476| 0.8829|
|20020412|    euro|3.40690476190476| 0.8792|
|20020415|    euro|3.40690476190476| 0.8802|
|20020416|    euro|3.40690476190476|  0.883|
|20020417|    euro|3.40690476190476| 0.8885|
|20020418|    euro|3.40690476190476| 0.8898|
|20020419|    euro|3.40690476190476| 0.8893|
|20020422|    euro|3.40690476190476| 0.8877|
|20020423|    euro|3.40690476190476| 0.8897|
|20020424|    euro|3.40690476190476| 0.8915|
|20020425|    euro|3.40690476190476| 0.8978|
|20020426|